# Lab - Unit 5: which steering does the real car prefer?

**Runs on your machine with CARLA 0.9.12 and `ROAR_PY`.**

The unit ended on a decision racekit cannot make.
On a car with no actuator delay, Unit 4's controller follows the line more closely than pure pursuit; with a delay of a couple of ticks at speed, Unit 4's controller goes unstable and pure pursuit does not.
`agents/unit-05.py` ships pure pursuit because CARLA's delay is unknown.
**In this lab the real car decides**: you race `agents/unit-04.py` and `agents/unit-05.py` on the official simulator, record each car's own cross-track error, and compare with racekit's predictions at delays of 0, 1 and 2 ticks.

How to use it:

- The cells tagged `no-exec` are for **your machine** (the Windows PC you set up in the Unit 1 lab).
  Their shell lines are shown with `!` only so the notebook stays valid.
- The analysis cells run anywhere.
  In CI they run against a racekit-made sample recording (the starter on `monza-like`), so the book always proves they work; on your machine they run against your own recordings, one at a time.

> **Never commit your results.**
> This repository is public.
> Your recordings, screenshots and logs stay on your own machine, **outside the repository**
> (the Unit 1 lab uses `%USERPROFILE%\roar-runs\`).
> The repository ignores data files under `labs/`, and a CI test fails if anything other than a notebook
> or a README is ever tracked there, but the first line of defence is not putting them here at all.

In [ ]:
import os

TELEMETRY_PATH = os.environ.get("ROAR_LAB_CSV", "telemetry.csv")

The parameters cell above is the **only** place a recording's file name appears.
Set `ROAR_LAB_CSV` to one recording's full path (outside the repository) before you start Jupyter; the recorder writes to the same variable.

## 1. The five steps

The Unit 1 lab (`units/unit-01-the-race-and-baseline/labs/real-sim.ipynb`) has every detail; this lab only changes what you race and what you record.

1. **Start the Monza v1.1 executable** (CARLA 0.9.12) and wait for the map to load (Unit 1 lab, §1 for the pinned versions and §3 for the start).
2. **Put the submission in the forked competition folder**: copy `agents/unit-04.py` over `ROAR_Competition\competition_code\submission.py`, paste in the Unit 1 recorder with its three wiring lines (Unit 1 lab, §2), and make the **one-line change** of §2 below.
   Before you go to CARLA, race the edited file on racekit and run `check_submission` on it, as the Unit 1 lab says.
3. **Run `competition_runner.py`** from `ROAR_Competition\competition_code`, with `ROAR_LAB_CSV` set to a new file outside the repository.
   Note the time and the collision count the runner reports.
4. **Export the telemetry to CSV**: the recorder does it as the car drives, one row per tick, to `ROAR_LAB_CSV`.
5. **Analyse it**: start Jupyter with `ROAR_LAB_CSV` pointing at the recording and run §3 onwards of this notebook.

Then do steps 2–5 again with `agents/unit-05.py`, to its own file.
Race each agent at least 3 times if you can: CARLA is not repeatable, and Unit 1's rule (median and range) applies here too.
Keep `TARGET_SPEED = 20.0` in both files: the comparison is at the speed racekit predicted.

## 2. The one-line recorder change

Unit 1's recorder writes 0 in the `offset` column, because the official runner does not tell the agent how far it is from the line.
But since Unit 2 your agent measures that itself: every `step()` calls `measure()`, which sets `self.cross_track_error` (Unit 2's windowed cross-track error, left positive) before the steering law runs.
So the recorder can write the agent's own measurement.
In `record()`, change the line that writes the driving row:

```diff
-            handle.write(",".join(f"{value:.6f}" for value in row) + f",{progress},0.000000,0\n")
+            handle.write(",".join(f"{value:.6f}" for value in row) + f",{progress},{self.cross_track_error:.6f},0\n")
```

(The comment above that line, "offset is 0", no longer applies; delete it if you like.)
The three wiring lines are the same as Unit 1's: `agents/unit-04.py` and `agents/unit-05.py` have the same `initialize()` and the same `self.current_index = self.advance_index(location)` line in `step()`, and `reanchor_after_respawn()` goes right after it, **before** `measure()` runs.

What the column now means: the cross-track error the agent measured at the **start** of that tick, from the position the sensors reported, against the official waypoints.
On racekit the same number is the telemetry's `offset` one row earlier (racekit writes the state *after* each tick); a routed test splices this exact change into `agents/unit-05.py`, races it, and checks the two agree to 1e-4 m.

**The stale row.** On CARLA the first `step()` after a respawn hold still sees the **crash pose** (Unit 1 lab, §2), so the first row after every hold run has `held = 0` but a cross-track error measured from where the car crashed, against the start of the lap.
That number is meaningless, and a single 5 m outlier would dominate an RMS of 0.2 m.
The analysis below drops it.

In [ ]:
# Anaconda Prompt, on the Windows machine, from ROAR_Competition\competition_code. Not run in CI.
# Type each line WITHOUT the leading "!" (it is there only so the notebook stays valid).
# Each run sets ROAR_LAB_CSV and starts the runner in ONE command, so the runner's process sees it;
# the quotes keep a trailing space out of the path.
# 1. Unit 4's agent (then paste the recorder in and make the one-line change)
!copy %PY4KIDS%\roar\projects\racekit\agents\unit-04.py submission.py
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-04-run-1.csv" && python competition_runner.py
# 2. Unit 5's agent, the same way, to its own recording
!copy %PY4KIDS%\roar\projects\racekit\agents\unit-05.py submission.py
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-05-run-1.csv" && python competition_runner.py
# 3. Analyse one recording: start Jupyter with the variable pointing at it (from your clone of this repository)
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-04-run-1.csv" && jupyter lab


## 3. Analyse one recording

Set `AGENT_NAME` to the agent this recording is from, and `RACE_FINISHED` to what the runner reported (Unit 1 lab, §4, explains why the recorder cannot know).

The statistics use the **driving rows**: not held (`held == 0`), and not the stale first row after a hold run.
A recording made without the hold rows (an older recorder) still shows a respawn as cumulative progress dropping to below 0.1·n, and then the row where it drops is the stale one.

In [ ]:
from math import isnan

import matplotlib.pyplot as plt
import numpy as np
from racekit.telemetry import read_csv, sector_splits

AGENT_NAME = "unit-05"  # "unit-04" or "unit-05": which agent this recording is from
RACE_FINISHED = True  # set False if the runner reported that the race did not finish


def driving_rows(telemetry, n):
    # The rows that count for the cross-track statistics, and the stale rows dropped:
    # not held, and not the first row after a hold run (on CARLA it still carries the crash pose).
    # Without held rows, a respawn shows as progress dropping below 0.1 * n; that row is the stale one.
    held = telemetry["held"] == 1
    keep = ~held
    held_rows = np.nonzero(held)[0]
    if len(held_rows):
        stale = [row + 1 for row in held_rows if row + 1 < len(held) and not held[row + 1]]
    else:
        progress = telemetry["progress"]
        stale = [row + 1 for row in np.nonzero((np.diff(progress) < 0) & (progress[1:] < 0.1 * n))[0]]
    keep[np.array(stale, dtype=int)] = False
    return keep, stale


telemetry, meta = read_csv(TELEMETRY_PATH)
n = meta["waypoints_per_lap"]
keep, stale = driving_rows(telemetry, n)
offset = telemetry["offset"][keep]
rms = float(np.sqrt(np.mean(offset ** 2)))
worst = float(np.max(np.abs(offset)))
if worst == 0.0:
    print("every offset is 0: was this recorded before the one-line change of section 2?")

held = telemetry["held"]
held_rows = np.nonzero(held)[0]
collisions = len([row for row in held_rows if row == 0 or not held[row - 1]]) if len(held_rows) else len(stale)
restart = float(telemetry["t"][held_rows[-1]]) if len(held_rows) else (
    float(telemetry["t"][stale[-1]]) if stale else 0.0)
splits = sector_splits(telemetry, meta["progress_total"], laps=meta["laps"], per_lap=meta["per_lap"],
                       finished=RACE_FINISHED)
complete = not any(isnan(split) for lap in splits for split in lap)
race_time = restart + sum(sum(lap) for lap in splits) if complete else float("nan")

print(f"{AGENT_NAME}: {len(telemetry)} rows, {int(np.sum(keep))} driving rows used ({len(stale)} stale rows dropped)")
print(f"RMS cross-track error {rms:.4f} m, max |offset| {worst:.3f} m")
print(f"collisions {collisions}, race time {race_time:.2f} s" if complete else f"collisions {collisions}, race incomplete")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(telemetry["t"][keep], telemetry["offset"][keep], linewidth=0.8)
ax.axhline(rms, color="grey", linestyle="--", linewidth=1.0)
ax.axhline(-rms, color="grey", linestyle="--", linewidth=1.0)
ax.set_xlabel("t (s)")
ax.set_ylabel("cross-track error (m, left +)")
ax.set_title(f"{AGENT_NAME}: the agent's own cross-track error (driving rows), RMS {rms:.3f} m dashed")
plt.show()

## 4. racekit's predictions

The same two files on racekit's `monza-like` at 20 m/s, with actuator jitter 0.05, seed 0, and an actuator delay of 0, 1 and 2 ticks (`KinematicCar(lag_ticks=...)`).
These six numbers are pinned in the plan (`roar-005`, U5.6), and this cell **asserts** them, so CI proves the table you compare against.
`monza-like` is inspired by Monza and is not the CARLA map, so compare the **pattern**, not the third decimal.
The six races take about half a minute.

In [ ]:
from pathlib import Path

from racekit.cars import KinematicCar
from racekit.runner import run_race

AGENTS = Path("..") / ".." / ".." / "projects" / "racekit" / "agents"
PREDICTED = {  # (agent, lag): (RMS cross-track error in m, race time in s), all clean
    ("unit-04", 0): (0.1054, 858.55), ("unit-04", 1): (0.1288, 858.95), ("unit-04", 2): (0.2420, 861.85),
    ("unit-05", 0): (0.1450, 858.60), ("unit-05", 1): (0.1655, 858.90), ("unit-05", 2): (0.2142, 859.75),
}
predicted = {}
for (agent, lag), (rms_pin, time_pin) in PREDICTED.items():
    result = run_race(AGENTS / f"{agent}.py", "monza-like", car=KinematicCar(jitter=0.05, lag_ticks=lag), seed=0)
    model_keep, _ = driving_rows(result.telemetry, result.progress_total // result.laps)
    model_rms = float(np.sqrt(np.mean(result.telemetry["offset"][model_keep] ** 2)))
    predicted[(agent, lag)] = (model_rms, result.total_time)
    assert result.finished and result.collisions == 0, (agent, lag, result)
    assert abs(model_rms - rms_pin) <= 0.0005 and abs(result.total_time - time_pin) <= 0.05 + 1e-9, (agent, lag)

print(f"{'racekit, monza-like, 20 m/s':30}{'lag 0':>14}{'lag 1':>14}{'lag 2':>14}")
for agent in ("unit-04", "unit-05"):
    row = "".join(f"{predicted[(agent, lag)][0]:8.4f} m    " for lag in range(3))
    print(f"{agent + ' RMS':30}{row}")
for agent in ("unit-04", "unit-05"):
    row = "".join(f"{predicted[(agent, lag)][1]:9.2f} s   " for lag in range(3))
    print(f"{agent + ' time':30}{row}")
print(f"\nyour recording ({AGENT_NAME}): RMS {rms:.4f} m, {collisions} collisions")

## 5. The crossover, and your decision

Read the table across.
With no delay Unit 4's controller is 4 cm closer to the line than pure pursuit; at one tick still 4 cm; at **two ticks the order flips**, and pure pursuit is 3 cm closer (and Unit 4's car has lost 3 s weaving).
So the two agents' RMS on the same car tells you which side of the crossover that car is on:

- **CARLA's Unit 4 RMS below its Unit 5 RMS** (and no collisions): the real car behaves like a small delay, at most about a tick.
  Unit 4's law is the better choice there: set `STEERING = "pid"` in your submission.
- **CARLA's Unit 5 RMS below its Unit 4 RMS, or Unit 4 collides**: the real car behaves like a delay of two ticks or more, and pure pursuit, the reference's choice, stays.

Record both agents' medians below (from at least 3 runs each, if you have them) and run the cell.
A difference smaller than the runs' range is a tie by Unit 1's rule: then keep pure pursuit, the choice that is robust to the delay you have not measured yet.
Unit 8 measures CARLA's delay directly.

In [ ]:
CARLA_RMS = {"unit-04": None, "unit-05": None}  # your CARLA medians (m), from this notebook's section 3
CARLA_RANGE = {"unit-04": None, "unit-05": None}  # their ranges over your runs (m), 0.0 for a single run
CARLA_COLLISIONS = {"unit-04": None, "unit-05": None}  # total collisions over your runs

if None in (CARLA_RMS["unit-04"], CARLA_RMS["unit-05"], CARLA_COLLISIONS["unit-04"]):
    print("fill in CARLA_RMS, CARLA_RANGE and CARLA_COLLISIONS from your recordings, then run this cell again")
else:
    gap = CARLA_RMS["unit-05"] - CARLA_RMS["unit-04"]
    noise = max(CARLA_RANGE["unit-04"] or 0.0, CARLA_RANGE["unit-05"] or 0.0)
    if CARLA_COLLISIONS["unit-04"] > 0 or gap < -noise:
        print("the real car behaves like a delay of two ticks or more: keep pure pursuit (STEERING = 'pursuit')")
    elif gap > noise:
        print("the real car behaves like a small delay: Unit 4's law follows the line better (STEERING = 'pid')")
    else:
        print("a tie within the noise: keep pure pursuit, the choice that is robust to an unknown delay")

## What to take away

- racekit's prediction is a **conditional** one: which controller is better depends on the delay, and the table shows where the answer flips.
  The real car's two RMS values put it on one side of that line.
- The agent's own cross-track error is a measurement, and a measurement has failure modes: the stale row after a respawn is one, and the analysis drops it on purpose.
- Whatever you decide, write down why, with the numbers: a choice made on one run of each agent is a guess.

> **One last check: never commit your results.**
> This repository is public.
> Your recordings, screenshots and logs stay on your own machine, in the folder outside the repository
> that `ROAR_LAB_CSV` pointed at.
> Run `git status` in your clone before you commit anything: none of them should be listed.